# Zero-Shot CoT v2 -- Output-Formatted Prompt -- Llama 3.1 8B (Colab)

**Motivation:** The vanilla Zero-Shot CoT prompt from Kojima 2022 produces
parse failures (the model states the correct answer but a unit modifier
follows it in the final sentence: *"Therefore Jim spends **36** hours in
**4** weeks"* -> parser grabs **4**).

This notebook fixes the issue **at the prompt level**: it asks the model
to end its response with `"The answer is X."`. That removes the parser
ambiguity entirely.

**Model:** `llama-3.1-8b-instant` (same 8B model as the original ZS-CoT)
**Sample:** 100 problems, `seed=42` (identical to the original ZS-CoT)
**Temperature:** 0.0 (greedy)

## Comparison plan
- **ZS-CoT v1 (vanilla):** 88.0% raw + 7 manual parse fixes -> 95.0%
- **ZS-CoT v2 (formatted):** ? -- measured by this notebook

Report **both versions** -- the cleaner v2 prompt formalises what the
manual re-validation already established.

## Prerequisites
1. **Secrets:** add `GROQ_API_KEY`
2. **Drive:** upload `MyDrive/NLP_ZSv2/data/gsm8k_test.json`

## Estimated time
- 100 x 1 call x 8s = **~13 min** (fast on 8B, same pace as ZS-CoT v1)


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE    = '/content/drive/MyDrive/NLP_ZSv2'
DRIVE_DATA    = os.path.join(DRIVE_BASE, 'data')
DRIVE_RESULTS = os.path.join(DRIVE_BASE, 'results')

os.makedirs(DRIVE_DATA,    exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print('Drive mounted.')
print('Data dir   :', DRIVE_DATA)
print('Results dir:', DRIVE_RESULTS)

In [ ]:
# 2. Install Groq SDK
!pip install groq -q
print('groq package ready.')

In [ ]:
# 3. Load API key
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY').strip()
if not GROQ_API_KEY:
    raise ValueError('GROQ_API_KEY not found. Add it from the Secrets panel.')
print('API key loaded.')

In [ ]:
# Configuration
N_SAMPLES   = 100
TEMPERATURE = 0.0   # greedy (same as ZS-CoT v1)
MODEL       = 'llama-3.1-8b-instant'
SEED        = 42

CHECKPOINT_FILE = os.path.join(DRIVE_RESULTS, 'zero_shot_cot_v2_results.json')
DATA_FILE       = os.path.join(DRIVE_DATA,    'gsm8k_test.json')

print(f'Target      : {N_SAMPLES} problems (seed={SEED})')
print(f'Model       : {MODEL}')
print(f'Temperature : {TEMPERATURE}')
print(f'Checkpoint  : {CHECKPOINT_FILE}')

In [ ]:
# 4. Data and checkpoint check
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f'Data file not found: {DATA_FILE}\n'
        'Please upload gsm8k_test.json to Drive/NLP_ZSv2/data/.'
    )
print('Data file found.')

import json as _json
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, encoding='utf-8') as _f:
        _ckpt = _json.load(_f)
    print(f'Existing checkpoint: {_ckpt.get("n_done", len(_ckpt.get("results", [])))} problems,'
          f' acc={_ckpt.get("accuracy")}%')
else:
    print('No checkpoint; will start from scratch.')

In [ ]:
# 5. LLM client (Groq)
import time
from groq import Groq

client = Groq(api_key=GROQ_API_KEY, timeout=120.0)

# Connectivity smoke test
try:
    _test = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': 'Say: hello'}],
        max_tokens=5,
    )
    print(f'Connectivity test ({MODEL}):', _test.choices[0].message.content)
except Exception as e:
    print(f'CONNECTIVITY TEST FAILED: {type(e).__name__}: {e}')
    raise


def chat(prompt, temperature=0.0, max_tokens=1024):
    for attempt in range(8):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            time.sleep(8)
            return response.choices[0].message.content.strip()
        except Exception as e:
            err_type = type(e).__name__
            print(f'\n  Error [{attempt+1}/8] ({err_type}): {str(e)[:200]}')
            wait = 60 * (attempt + 1) if 'RateLimit' in err_type or '429' in str(e) else 15
            if attempt < 7:
                print(f'  waiting {wait}s...')
                time.sleep(wait)
    raise RuntimeError('Failed after 8 attempts.')

print('LLM client ready.')

In [ ]:
# 6. ZS-CoT v2 prompt template (output-format hint)

PROMPT_TEMPLATE = (
    "Q: {question}\n"
    "A: Let's think step by step. "
    "At the end, write your final numeric answer in the format: "
    "\"The answer is X.\" where X is just a single number with no units."
)

print('Prompt template ready.')
print('\nSample prompt:')
print(PROMPT_TEMPLATE.format(question='How many apples does Tim have if he buys 3 then loses 1?'))

In [ ]:
# 7. Load data (seed=42 -> identical 100 problems to ZS-CoT v1 / FS-CoT / SC)
import json
import random

with open(DATA_FILE, encoding='utf-8') as f:
    all_data = json.load(f)

random.seed(SEED)
test_data = random.sample(all_data, N_SAMPLES)

print(f'Total test set: {len(all_data)}')
print(f'This run      : {len(test_data)} problems (seed={SEED})')

In [ ]:
# 8. Helpers -- the new parser prefers the 'answer is X' pattern

import re

def extract_answer(text):
    """
    Precedence:
      1. GSM8K '#### X' marker
      2. 'The answer is X' / 'answer is X' (case-insensitive, last occurrence)
      3. Fallback: last number in text
    """
    # 1) #### marker
    m = re.search(r'####\s*\$?([\d,]+\.?\d*)', text)
    if m:
        try: return float(m.group(1).replace(',', ''))
        except: pass
    # 2) 'answer is X' (case-insensitive, last occurrence)
    matches = re.findall(
        r'(?:final\s+)?answer\s+is\s*[:\-]?\s*\$?([\d,]+\.?\d*)',
        text, re.IGNORECASE
    )
    if matches:
        try: return float(matches[-1].replace(',', ''))
        except: pass
    # 3) Fallback: last number
    nums = re.findall(r'[\d,]+\.?\d*', text.replace(',', ''))
    return float(nums[-1]) if nums else None


def gold_answer(text):
    m = re.search(r'####\s*([\d,\.]+)', text)
    return float(m.group(1).replace(',', '')) if m else None


print('Helpers ready.')

In [ ]:
# 9. ZS-CoT v2 -- run + checkpoint

results        = []
done_questions = set()

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, encoding='utf-8') as f:
        ckpt = json.load(f)
    results        = ckpt.get('results', [])
    done_questions = {r['question'] for r in results}
    print(f'Checkpoint found: {len(results)} problems already done.')
else:
    print('No checkpoint; starting from scratch.')

total = len(test_data)

for i, item in enumerate(test_data, 1):
    if item['question'] in done_questions:
        print(f'[{i}/{total}] skipping (already done)', end='\r', flush=True)
        continue

    prompt = PROMPT_TEMPLATE.format(question=item['question'])
    print(f'[{i}/{total}] solving...', end='\r', flush=True)

    response  = chat(prompt, temperature=TEMPERATURE)
    predicted = extract_answer(response)
    gold      = gold_answer(item['answer'])

    results.append({
        'question':  item['question'],
        'gold':      gold,
        'predicted': predicted,
        'correct':   predicted == gold,
        'response':  response,
    })

    correct_so_far = sum(r['correct'] for r in results)
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'strategy':       'zero_shot_cot_v2_output_formatted',
            'model':          MODEL,
            'temperature':    TEMPERATURE,
            'prompt_format':  'Let\'s think step by step + answer is X format hint',
            'n_done':         len(results),
            'accuracy':       round(correct_so_far / len(results) * 100, 1),
            'results':        results,
        }, f, indent=2, ensure_ascii=False)

correct = sum(r['correct'] for r in results)
n       = len(results)
print(f'\nZS-CoT v2 (output-formatted) finished.')
print(f'Result: {correct}/{n} correct -- Accuracy = {correct/n*100:.1f}%')

In [ ]:
# (Optional) 10. Comparison summary
print(f'\n=== ZS-CoT comparison (after gold-correction) ===')
print(f'v1 vanilla prompt            : 88.0% raw + 7 manual fixes -> 95.0%')
print(f'v2 output-formatted (this run): {correct/n*100:.1f}% raw')

# Carnival gold-correction
gc = correct
for r in results:
    if 'carnival' in r.get('question','').lower():
        print(f'\nCarnival (annotation error):')
        print(f'  Gold (incorrect): {r["gold"]}')
        print(f'  Predicted       : {r["predicted"]}')
        if r['predicted'] == 2180.0 and not r['correct']:
            gc += 1
            print(f'  -> Gold-correction adds +1 correct')
        break

print(f'\nv2 gold-corrected: {gc}/{n} = {100*gc/n:.1f}%')

print('\n=== Expected outcomes ===')
delta = gc/n*100 - 95.0
if abs(delta) <= 2:
    print(f'  delta = {delta:+.1f} pp -> CONFIRMS the manual re-validation')
elif delta > 2:
    print(f'  delta = {delta:+.1f} pp -> formatted prompt gives ADDITIONAL gain')
else:
    print(f'  delta = {delta:+.1f} pp -> manual fix overestimated; formatted prompt is more ROBUST')

# Format compliance check (did the model honor the 'answer is' instruction?)
print('\n=== Format compliance ===')
import re
n_compliant = sum(1 for r in results
                  if re.search(r'answer\s+is', r.get('response',''), re.IGNORECASE))
print(f'Responses with "answer is" pattern: {n_compliant}/{n} = {100*n_compliant/n:.0f}%')